# 02 — SQL Analysis · Olist Brazilian E-Commerce

Business questions answered directly in **SQL** on the Olist SQLite database
(11 tables, `orders` at the center).

Each section: **question → query → result → short read**.
The queries are also stored as `.sql` files in `../sql/`.

## 0 · Setup

Connect to the database. `q("...")` runs a SQL string and returns a table (DataFrame).

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

# find the database (works locally and on Kaggle)
_db = [Path("../data/raw/olist.sqlite"), Path("data/raw/olist.sqlite"),
       Path("/kaggle/input/brazilian-ecommerce/olist.sqlite")]
con = sqlite3.connect(next((p for p in _db if p.exists()), _db[0]))

def q(sql):
    return pd.read_sql_query(sql, con)

# list all tables
q("SELECT name AS table_name FROM sqlite_master WHERE type='table' ORDER BY name")

,table_name
0,customers
1,geolocation
2,leads_closed
3,leads_qualified
4,order_items
5,order_payments
6,order_reviews
7,orders
8,product_category_name_translation
9,products


## 1 · Revenue over time

**Question:** How many orders and how much revenue per month?

**Query in short**
- `JOIN order_items` — the price is stored there, so we attach it to each order
- `GROUP BY month` — one row per month
- `COUNT(DISTINCT order_id)` — number of orders (an order can have several items)
- `SUM(price)` — revenue

In [2]:
# Same query is stored in sql/01_revenue_over_time.sql
rev = q("""
-- 01 · Revenue over time (Umsatz über Zeit)
-- Frage: Wie viele Bestellungen und wie viel Umsatz gibt es pro Monat?

SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS month,    -- Monat, z.B. "2018-05"
    COUNT(DISTINCT o.order_id)                     AS orders,   -- Anzahl Bestellungen
    ROUND(SUM(oi.price), 2)                         AS revenue   -- Umsatz = Summe der Preise
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id                -- Preise zu den Bestellungen holen
GROUP BY month                                                  -- eine Zeile pro Monat
ORDER BY month;
""")
rev

,month,orders,revenue
0,2016-09,3,267.36
1,2016-10,308,49507.66
2,2016-12,1,10.90
3,2017-01,789,120312.87
4,2017-02,1733,247303.02
5,2017-03,2641,374344.30
6,2017-04,2391,359927.23
7,2017-05,3660,506071.14
8,2017-06,3217,433038.60
9,2017-07,3969,498031.48


**Read:** 24 months (2016-09 → 2018-09), total revenue **R$ 13.59 M** — the same number
as the KPI in the Power BI dashboard. Revenue reaches roughly R$ 1 M per month in 2018.
The edge months are thin (2018-09 has only 1 order), typical for the Olist dataset.

## 2 · Top categories

**Question:** Which product categories generate the most revenue?

**Query in short**
- `order_items → products` — the item knows its product, the product knows its category
- `JOIN` the translation table — gives the English category name
- `ORDER BY revenue DESC` + `LIMIT 10` — the 10 biggest categories

*Note:* products without a category translation are left out (about 1.4 % of revenue).

In [3]:
# Same query is stored in sql/02_top_categories.sql
top_cat = q("""
-- 02 · Top categories (Top-Kategorien)
-- Frage: Welche Produktkategorien bringen den meisten Umsatz?

SELECT
    t.product_category_name_english AS category,   -- Kategorie auf Englisch
    COUNT(DISTINCT oi.order_id)     AS orders,      -- Anzahl Bestellungen
    ROUND(SUM(oi.price), 2)         AS revenue      -- Umsatz = Summe der Preise
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id                          -- Artikel -> Produkt
JOIN product_category_name_translation t
     ON t.product_category_name = p.product_category_name                 -- Produkt -> englischer Name
GROUP BY category                                                         -- eine Zeile pro Kategorie
ORDER BY revenue DESC                                                     -- größter Umsatz zuerst
LIMIT 10;                                                                 -- nur die Top 10

-- Hinweis: Produkte ohne Kategorie-Übersetzung fallen hier heraus (ca. 1,4 % des Umsatzes).
""")
top_cat

,category,orders,revenue
0,health_beauty,8836,1258681.34
1,watches_gifts,5624,1205005.68
2,bed_bath_table,9417,1036988.68
3,sports_leisure,7720,988048.97
4,computers_accessories,6689,911954.32
5,furniture_decor,6449,729762.49
6,cool_stuff,3632,635290.85
7,housewares,5884,632248.66
8,auto,3897,592720.11
9,garden_tools,3518,485256.46


**Read:** `health_beauty` and `watches_gifts` lead by revenue.
Order count and revenue rank differently: `bed_bath_table` has the most orders (9,417)
but only ranks 3rd by revenue, while `watches_gifts` earns more with fewer orders (5,624) —
it sells higher-priced items.

## 3 · Orders by state

**Question:** Which Brazilian states place the most orders?

**Query in short**
- The state is stored in `customers`, not in `orders` → `JOIN customers` via `customer_id`
- The price is stored in `order_items` → `JOIN order_items` via `order_id`
- `GROUP BY state` — one row per state

In [4]:
# Same query is stored in sql/03_orders_by_state.sql
by_state = q("""
-- 03 · Orders by state (Bestellungen nach Bundesland)
-- Frage: Aus welchen Bundesländern kommen die meisten Bestellungen?

SELECT
    c.customer_state           AS state,     -- Bundesland, z.B. SP = São Paulo
    COUNT(DISTINCT o.order_id) AS orders,    -- Anzahl Bestellungen
    ROUND(SUM(oi.price), 2)    AS revenue    -- Umsatz = Summe der Preise
FROM orders o
JOIN customers c    ON c.customer_id = o.customer_id    -- das Bundesland steht beim Kunden
JOIN order_items oi ON oi.order_id   = o.order_id       -- der Preis steht bei den Artikeln
GROUP BY state                                           -- eine Zeile pro Bundesland
ORDER BY orders DESC                                     -- meiste Bestellungen zuerst
LIMIT 10;                                                -- nur die Top 10
""")
by_state

,state,orders,revenue
0,SP,41375,5202955.05
1,RJ,12762,1824092.67
2,MG,11544,1585308.03
3,RS,5432,750304.02
4,PR,4998,683083.76
5,SC,3612,520553.34
6,BA,3358,511349.99
7,DF,2125,302603.94
8,ES,2025,275037.31
9,GO,2007,294591.95


**Read:** The market is very concentrated. **São Paulo (SP) has 41,375 orders** — more than
RJ (12,762) and MG (11,544) combined (24,306). The same pattern shows in revenue:
SP earns R$ 5.2 M, more than the next three states together.

## 4 · Delivery performance vs. reviews

**Question:** Do customers rate worse when the delivery arrives late?

**Query in short**
- `CASE WHEN delivered <= estimated THEN 'on time' ELSE 'late'` — sorts each order into one of two groups
- `JOIN order_reviews` — attaches the review score to each order
- `WHERE order_status = 'delivered'` — only orders that were actually delivered
- `AVG(review_score)` — average stars per group
- `julianday(a) - julianday(b)` — difference between two dates in days

In [5]:
# Same query is stored in sql/04_delivery_vs_reviews.sql
delivery = q("""
-- 04 · Delivery performance vs. reviews (Lieferung und Bewertung)
-- Frage: Bewerten Kunden schlechter, wenn die Lieferung zu spät kommt?

SELECT
    -- pünktlich = geliefert am oder vor dem versprochenen Termin, sonst verspätet
    CASE WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
         THEN 'on time' ELSE 'late' END                      AS delivery,
    COUNT(DISTINCT o.order_id)                               AS orders,            -- Anzahl Bestellungen
    ROUND(AVG(r.review_score), 2)                            AS avg_review_score,  -- Ø Sterne (1 bis 5)
    ROUND(AVG(julianday(o.order_delivered_customer_date)
            - julianday(o.order_purchase_timestamp)), 1)     AS avg_delivery_days  -- Ø Tage von Kauf bis Lieferung
FROM orders o
JOIN order_reviews r ON r.order_id = o.order_id              -- Bewertung zur Bestellung holen
WHERE o.order_status = 'delivered'                           -- nur tatsächlich gelieferte Bestellungen
  AND o.order_delivered_customer_date IS NOT NULL            -- mit bekanntem Lieferdatum
GROUP BY delivery                                            -- eine Zeile für 'on time', eine für 'late'
ORDER BY delivery DESC;
""")
delivery

,delivery,orders,avg_review_score,avg_delivery_days
0,on time,88163,4.29,10.9
1,late,7661,2.57,31.4


**Read:** Late deliveries hurt ratings a lot. Orders delivered **on time** average
**4.29 stars**, **late** orders only **2.57**. Late orders are about 8 % of all delivered
orders (7,661 of roughly 95,800) and take on average **31.4 days** instead of 10.9.

This confirms the strongest signal from the EDA (delivery time ↔ review score) and is the
basis for the ML notebook: predicting bad reviews. *Note:* this shows a strong link, not
proof of cause — late orders also tend to be the long, problematic ones.

## 5 · Payment methods

**Question:** How do customers pay, and how often do they use installments?

**Query in short**
- Everything is in one table (`order_payments`), so no `JOIN` is needed
- `GROUP BY payment_type` — one row per payment method
- `AVG(payment_installments)` — average number of installments

*Note:* `payment_value` includes shipping costs, so the total is higher than the revenue
from section 1 (item prices only). An order can also be paid with several methods,
so the order counts add up to slightly more than the number of orders.

In [6]:
# Same query is stored in sql/05_payment_methods.sql
payments = q("""
-- 05 · Payment methods (Zahlungsarten)
-- Frage: Wie bezahlen Kunden, und wie oft nutzen sie Ratenzahlung?

SELECT
    payment_type,                                          -- Zahlungsart, z.B. credit_card, boleto
    COUNT(DISTINCT order_id)            AS orders,           -- Anzahl Bestellungen
    ROUND(SUM(payment_value), 2)        AS total_value,      -- gezahlte Summe
    ROUND(AVG(payment_installments), 1) AS avg_installments  -- Ø Anzahl Raten
FROM order_payments                                          -- hier reicht eine Tabelle, kein JOIN nötig
GROUP BY payment_type                                        -- eine Zeile pro Zahlungsart
ORDER BY orders DESC;                                        -- häufigste Zahlungsart zuerst

-- Hinweis: payment_value enthält auch die Versandkosten, deshalb ist die
-- Summe höher als der Umsatz aus Abfrage 1 (nur Artikelpreise).
""")
payments

,payment_type,orders,total_value,avg_installments
0,credit_card,76505,12542084.19,3.5
1,boleto,19784,2869361.27,1.0
2,voucher,3866,379436.87,1.0
3,debit_card,1528,217989.79,1.0
4,not_defined,3,0.00,1.0


**Read:** **Credit card** dominates with 76,505 orders — almost four times as many as
**boleto** (19,784), the Brazilian bank-slip payment. Credit-card customers use
**3.5 installments on average**, while all other methods are paid at once (1.0).
Installments are a big part of how Brazilians buy online.

## 6 · Top sellers

**Question:** Which sellers generate the most revenue, and where are they based?

**Query in short**
- `order_items` holds the seller and the price; `JOIN sellers` adds the seller's state
- `GROUP BY seller_id` — one row per seller
- `ORDER BY revenue DESC` + `LIMIT 10` — the 10 biggest sellers

In [7]:
# Same query is stored in sql/06_top_sellers.sql
sellers = q("""
-- 06 · Top sellers (beste Verkäufer)
-- Frage: Welche Verkäufer machen den meisten Umsatz, und wo sitzen sie?

SELECT
    oi.seller_id,                                      -- Verkäufer-ID
    s.seller_state              AS state,                -- Bundesland des Verkäufers
    COUNT(DISTINCT oi.order_id) AS orders,               -- Anzahl Bestellungen
    ROUND(SUM(oi.price), 2)     AS revenue               -- Umsatz = Summe der Preise
FROM order_items oi
JOIN sellers s ON s.seller_id = oi.seller_id             -- Verkäufer-Daten (Bundesland) dazuholen
GROUP BY oi.seller_id                                    -- eine Zeile pro Verkäufer
ORDER BY revenue DESC                                    -- größter Umsatz zuerst
LIMIT 10;                                                -- nur die Top 10
""")
sellers

,seller_id,state,orders,revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1132,229472.63
1,53243585a1d6dc2643021fd1853d8905,BA,358,222776.05
2,4a3ca9315b744ce9f8e9374361493884,SP,1806,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,585,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,SP,982,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,SP,336,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,1314,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,SP,1160,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,915,138968.55
9,955fee9216a65b617aa5c0531780ce60,SP,1287,135171.70


**Read:** Revenue is spread across **3,095 sellers**; the top 10 together make up about
**13 %** of all revenue, so no single seller dominates. **9 of the top 10 are based in
São Paulo**, matching the customer concentration from section 3. Order count and revenue
differ again: the seller with the most orders (1,806) ranks only 3rd by revenue — lower
prices per item.